# Functional API

This notebook accompanies the [Agent Foundry](https://agent-foundry-pi.vercel.app) LangGraph roadmap.

You will learn the Functional API alternative to StateGraph — using `@entrypoint` and `@task` decorators with standard Python control flow.

## 1. Install Dependencies

In [ ]:
!pip install -q langgraph langchain "langchain[openai]"

## 2. Set Up Your API Key

Enter your OpenAI API key when prompted. The key is not stored or displayed.

In [ ]:
import os
from getpass import getpass

os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")

## 3. A Simple Workflow with @entrypoint and @task

Tasks return futures — call `.result()` to get the value.

In [ ]:
from langgraph.func import entrypoint, task
from langchain_openai import ChatOpenAI

llm = ChatOpenAI(model="gpt-4o-mini")

@task
def write_joke(topic: str) -> str:
    response = llm.invoke(f"Write a short joke about {topic}")
    return response.content

@task
def rate_joke(joke: str) -> str:
    response = llm.invoke(f"Rate this joke 1-10 and explain: {joke}")
    return response.content

@entrypoint()
def joke_workflow(topic: str) -> dict:
    joke = write_joke(topic).result()
    rating = rate_joke(joke).result()
    return {"joke": joke, "rating": rating}

result = joke_workflow.invoke("AI agents")
print(f"Joke: {result['joke']}")
print(f"Rating: {result['rating']}")

## 4. Control Flow with Python Loops

Use standard `for`/`while`/`if` instead of conditional edges.

In [ ]:
@entrypoint()
def iterative_writer(topic: str) -> dict:
    joke = write_joke(topic).result()
    for attempt in range(3):
        rating = rate_joke(joke).result()
        if any(score in rating for score in ["8", "9", "10"]):
            return {"joke": joke, "rating": rating, "attempts": attempt + 1}
        joke = write_joke(topic).result()
    return {"joke": joke, "rating": rating, "attempts": 3}

result = iterative_writer.invoke("programming")
print(f"Joke: {result['joke']}")
print(f"Rating: {result['rating']}")
print(f"Attempts: {result['attempts']}")

## 5. Rebuild the Tool-Calling Agent

The same agent loop from the Graph API topic, using plain Python instead of edges.

In [ ]:
from langchain_core.tools import tool
from langchain_core.messages import HumanMessage, ToolMessage

@tool
def add(a: float, b: float) -> float:
    """Add two numbers together."""
    return a + b

@tool
def multiply(a: float, b: float) -> float:
    """Multiply two numbers together."""
    return a * b

tools = [add, multiply]
tools_by_name = {t.name: t for t in tools}

llm_with_tools = llm.bind_tools(tools)

@task
def call_llm(messages: list) -> dict:
    response = llm_with_tools.invoke(messages)
    return response

@task
def call_tool(tool_call: dict) -> str:
    tool_fn = tools_by_name[tool_call["name"]]
    result = tool_fn.invoke(tool_call["args"])
    return ToolMessage(content=str(result), tool_call_id=tool_call["id"])

@entrypoint()
def agent(question: str) -> str:
    messages = [HumanMessage(content=question)]

    while True:
        response = call_llm(messages).result()
        messages.append(response)

        if not response.tool_calls:
            return response.content

        for tc in response.tool_calls:
            tool_result = call_tool(tc).result()
            messages.append(tool_result)

result = agent.invoke("What is 15 * 3 + 20?")
print(f"Answer: {result}")

## 6. Try More Questions

In [ ]:
questions = [
    "What is 8 + 12?",
    "What is 7 * 6 + 2?",
    "Multiply 100 by 5 and add 50",
]

for q in questions:
    answer = agent.invoke(q)
    print(f"Q: {q}")
    print(f"A: {answer}\n")

## What You Learned

1. **`@entrypoint`** marks the main workflow function — replaces `graph.compile()`
2. **`@task`** marks sub-operations that return futures resolved with `.result()`
3. Standard Python **loops and conditionals** replace conditional edges
4. The tool-calling agent loop becomes a simple **`while True`** with an `if` break
5. Both APIs support **checkpointing, streaming, and tracing**